In [1]:
from pathlib import Path
import gc
import time

import cudf
import cupy as cp
import numpy as np
from cuml.neighbors import NearestNeighbors
from cuml.preprocessing import PowerTransformer, RobustScaler, StandardScaler

RANDOM_STATE = 42
K = 5
REFERENCE_SIZE = 20_000
BATCH_SIZE = 2_000
MISSING_FLAG_THRESHOLD = 0.05
EXTREME_VALUE_LIMIT = 1e30
REFERENCE_DATE = cudf.to_datetime("2017-12-01")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
REFERENCE_UNIX_SECONDS = 1_512_086_400
TRAIN_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = DATA_DIR / "train_identity.csv"
OUTPUT_DIR = PROJECT_ROOT / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "train_preprocessed.csv"
IDS_OUTPUT_PATH = OUTPUT_DIR / "train_ids.csv"

print("cuDF:", cudf.__version__)
print("CuPy:", cp.__version__)
try:
    import cuml
    print("cuML:", cuml.__version__)
    device = cp.cuda.Device()
    free_mem, total_mem = cp.cuda.runtime.memGetInfo()
    print(f"GPU {device.id}: {total_mem / 2**30:.2f} GB total, {free_mem / 2**30:.2f} GB free")
except Exception as exc:
    raise RuntimeError("RAPIDS and a working NVIDIA CUDA device are required for this notebook.") from exc


cuDF: 26.06.01
CuPy: 14.2.0
cuML: 26.06.00
GPU 0: 6.00 GB total, 4.95 GB free


In [2]:
train_transaction = cudf.read_csv(str(TRAIN_TRANSACTION_PATH))
train_identity = cudf.read_csv(str(TRAIN_IDENTITY_PATH))
df = train_transaction.merge(train_identity, on="TransactionID", how="left").reset_index(drop=True)
del train_transaction, train_identity
gc.collect()

print("Merged shape:", df.shape)
print("Target distribution:")
print(df["isFraud"].value_counts())


Merged shape: (590540, 434)
Target distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64


In [3]:
if "TransactionDT" in df:
    elapsed = cudf.to_datetime(df["TransactionDT"], unit="s")
    transaction_time = cudf.to_datetime(
    df["TransactionDT"].astype("int64") + REFERENCE_UNIX_SECONDS,
    unit="s",
    )
    df["transaction_hour"] = transaction_time.dt.hour.astype("float32")
    df["transaction_dayofweek"] = transaction_time.dt.dayofweek.astype("float32")
    df["transaction_month"] = transaction_time.dt.month.astype("float32")
    df["transaction_year"] = transaction_time.dt.year.astype("float32")
    df["transaction_day"] = transaction_time.dt.day.astype("float32")
    hours = transaction_time.dt.hour
    df["transaction_part_of_day"] = cudf.Series("evening", index=df.index, dtype="str")
    df.loc[hours < 6, "transaction_part_of_day"] = "night"
    df.loc[(hours >= 6) & (hours < 12), "transaction_part_of_day"] = "morning"
    df.loc[(hours >= 12) & (hours < 18), "transaction_part_of_day"] = "afternoon"
    df["is_weekend"] = (transaction_time.dt.dayofweek >= 5).astype("int8")

card_keys = [c for c in ["card1", "card2", "card3", "card5", "card6"] if c in df.columns]
if card_keys:
    # The source transaction table is time ordered; cumcount therefore uses past rows.
    df["card_velocity"] = df.groupby(card_keys).cumcount().astype("float32")
    if "TransactionAmt" in df and "card1" in df:
        card_average = df.groupby("card1")["TransactionAmt"].transform("mean")
        df["amt_to_avg_card_ratio"] = df["TransactionAmt"] / card_average.replace(0, None)
    if "addr1" in df and "card1" in df:
        card_addr = (df["card1"].astype("str").fillna("Unknown") + "|" +
                     df["addr1"].astype("str").fillna("Unknown"))
        df["card_addr_hash"] = card_addr.astype("category").cat.codes.astype("float32")
        if "amt_to_avg_card_ratio" in df:
            df["card_addr_amt_ratio"] = df["card_addr_hash"] * df["amt_to_avg_card_ratio"]

crucial_cols = [c for c in ["card1", "card2", "card3", "card4", "card5", "card6",
                            "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceInfo", "DeviceType"]
                if c in df.columns]
if crucial_cols:
    df["missing_crucial_info"] = df[crucial_cols].isna().astype("int8").sum(axis=1)

# Add sign-preserving log1p features for strongly skewed numeric columns.
base_numeric = [c for c in df.select_dtypes(include=["number"]).columns if c not in ["isFraud", "TransactionID"]]
for col in base_numeric:
    values = df[col]
    if int(values.count()) > 2 and abs(float(values.skew())) > 1.0:
        df[f"{col}_log1p"] = cp.sign(values.to_cupy()) * cp.log1p(cp.abs(values.to_cupy()))

print("Shape after GPU feature engineering:", df.shape)


Shape after GPU feature engineering: (590540, 813)


In [4]:
TARGET = "isFraud"
ids = df[["TransactionID"]].copy() if "TransactionID" in df else cudf.DataFrame(index=df.index)
y = df[TARGET].copy() if TARGET in df else None
df = df.drop(columns=[c for c in [TARGET, "TransactionID"] if c in df.columns])

v_features = [c for c in df.columns if c.startswith("V") and c[1:].isdigit()]
v_drop = [c for c in v_features if float(df[c].isna().mean()) > 0.90]
df = df.drop(columns=v_drop)
v_features = [c for c in v_features if c not in v_drop]

known_categorical = {"ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
                     "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceInfo", "DeviceType"}
known_categorical.update(f"M{i}" for i in range(1, 10))
known_categorical.update(f"id_{i}" for i in range(12, 39))

numeric_cols = list(df.select_dtypes(include=["number"]).columns)
for col in numeric_cols:
    values = df[col].replace([np.inf, -np.inf], None)
    df[col] = values.where(values.abs() <= EXTREME_VALUE_LIMIT)

cat_cols = [c for c in df.columns if c in known_categorical or str(df[c].dtype) in ("object", "str", "category") ]
numeric_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in cat_cols]
flag_cols = []
for col in cat_cols + numeric_cols:
    rate = float(df[col].isna().mean())
    create_flag = rate > MISSING_FLAG_THRESHOLD or (col.startswith(("C", "D", "id_")) and rate > 0)
    if create_flag:
        flag = f"{col}_missing"
        df[flag] = df[col].isna().astype("int8")
        flag_cols.append(flag)

cat_cols = [c for c in cat_cols if c in df.columns]
numeric_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in cat_cols]
print("Removed V-features (>90% missing):", len(v_drop))
print("Categorical features:", len(cat_cols), "Numeric features:", len(numeric_cols))
print("Missing indicators:", len(flag_cols))


Removed V-features (>90% missing): 0
Categorical features: 50 Numeric features: 1344
Missing indicators: 583


In [5]:
# Convert categorical feature values to explicit strings before ordinal encoding.
for col in cat_cols:
    df[col] = df[col].astype("str").fillna("Unknown")
    df[col] = df[col].where(df[col] != "<NA>", "Unknown")

missing_rate = df[numeric_cols].isna().mean()
knn_cols = [c for c in numeric_cols if 0 < float(missing_rate[c]) < 0.50]
median_cols = [c for c in numeric_cols if float(missing_rate[c]) >= 0.50]
knn_models = {}

def numeric_family(col):
    if col.startswith("V") and col[1:].split("_")[0].isdigit(): return "V"
    if col.startswith("C") and col[1:].split("_")[0].isdigit(): return "C"
    if col.startswith("D") and col[1:].split("_")[0].isdigit(): return "D"
    if col.startswith("id_"): return "ID"
    return "OTHER"

def gpu_knn_impute_columns(frame, columns, family):
    complete_mask = frame[columns].notna().all(axis=1)
    complete_count = int(complete_mask.sum())
    if complete_count < K:
        print(f"{family}: only {complete_count} complete reference rows; median fallback")
        for col in columns:
            median = frame[col].median()
            frame[col] = frame[col].fillna(0 if median is None else median)
        return frame, None

    reference = frame.loc[complete_mask, columns].astype("float32")
    n_reference = min(REFERENCE_SIZE, len(reference))
    if len(reference) > n_reference:
        reference = reference.sample(n=n_reference, random_state=RANDOM_STATE)
    reference_gpu = cp.asarray(reference.to_cupy(), dtype=cp.float32)
    reference_mean = cp.mean(reference_gpu, axis=0)
    scaler = StandardScaler()
    reference_scaled = scaler.fit_transform(reference_gpu)
    model = NearestNeighbors(n_neighbors=K, metric="euclidean")
    model.fit(reference_scaled)

    missing_mask = frame[columns].isna().any(axis=1)
    missing_positions = cp.flatnonzero(missing_mask.to_cupy()).get().tolist()
    for start in range(0, len(missing_positions), BATCH_SIZE):
        batch_positions = missing_positions[start:start + BATCH_SIZE]
        batch_gpu = cp.asarray(frame.iloc[batch_positions][columns].astype("float32").to_cupy())
        cell_mask = cp.isnan(batch_gpu)
        query = cp.where(cell_mask, reference_mean[None, :], batch_gpu)
        query_scaled = scaler.transform(query)
        _, neighbor_indices = model.kneighbors(query_scaled, n_neighbors=K)
        neighbor_indices = cp.asarray(neighbor_indices, dtype=cp.int32)
        neighbor_values = reference_gpu[neighbor_indices]
        neighbor_means = cp.mean(neighbor_values, axis=1)
        # Preserve observed entries; replace missing cells with neighbor means only.
        result = cp.where(cell_mask, neighbor_means, batch_gpu)
        for col_index, col in enumerate(columns):
            frame[col].iloc[batch_positions] = result[:, col_index]
        del batch_gpu, query, query_scaled, neighbor_values, neighbor_means, result
        cp.get_default_memory_pool().free_all_blocks()
        gc.collect()
    return frame, (reference_gpu, scaler, model, reference_mean)

for family in sorted({numeric_family(c) for c in knn_cols}):
    family_cols = [c for c in knn_cols if numeric_family(c) == family]
    df, fitted = gpu_knn_impute_columns(df, family_cols, family)
    if fitted is not None:
        knn_models[family] = fitted

for col in median_cols:
    median = df[col].median()
    df[col] = df[col].fillna(0 if median is None else median)

print("GPU KNN columns:", len(knn_cols))
print("Median columns:", len(median_cols))
print("Remaining missing cells before categorical encoding:", int(df.isna().sum().sum()))


GPU KNN columns: 355
Median columns: 362
Remaining missing cells before categorical encoding: 0


In [6]:
frequency_cols = [c for c in ["card1", "card2", "card3"] if c in cat_cols]
for col in frequency_cols:
    frequencies = df[col].value_counts(normalize=True)
    df[f"{col}_freq"] = df[col].map(frequencies).fillna(0).astype("float32")

ordinal_cols = [c for c in cat_cols if c not in frequency_cols]
for col in ordinal_cols:
    df[col] = df[col].astype("category").cat.codes.astype("float32")

# Ordinal codes stay categorical; the frequency features and numeric fields are scaled.
scale_cols = [c for c in df.select_dtypes(include=["number"]).columns if c not in ordinal_cols]
scale_cols = [c for c in scale_cols if int(df[c].nunique()) > 1]
winsor_bounds = {}
for col in scale_cols:
    q = df[col].quantile([0.01, 0.99])
    lower, upper = float(q.loc[0.01]), float(q.loc[0.99])
    winsor_bounds[col] = (lower, upper)
    df[col] = df[col].clip(lower=lower, upper=upper)

# Yeo-Johnson is unsuitable for constant/two-valued columns, and its lambda
# optimizer may fail to bracket a solution for some discrete/skewed features.
skewed_cols = [
    c for c in scale_cols
    if int(df[c].nunique()) > 2 and abs(float(df[c].skew())) > 1.0
]
power_transformers = {}
power_fallbacks = []
for col in skewed_cols:
    try:
        transformer = PowerTransformer(method="yeo-johnson", standardize=False)
        transformed = transformer.fit_transform(df[[col]].astype("float32").to_cupy())
        df[col] = cudf.Series(cp.asarray(transformed).reshape(-1), index=df.index)
        power_transformers[col] = transformer
    except Exception as exc:
        # Keep the pipeline moving: apply a GPU signed-log transform if YJ
        # cannot estimate a stable lambda for this feature.
        values = df[col].to_cupy()
        df[col] = cudf.Series(cp.sign(values) * cp.log1p(cp.abs(values)), index=df.index)
        power_fallbacks.append((col, type(exc).__name__))
        print(f"Yeo-Johnson fallback for {col}: {type(exc).__name__}; applied signed log1p")

robust_scaler = RobustScaler()
scaled = robust_scaler.fit_transform(df[scale_cols].astype("float32").to_cupy())
for i, col in enumerate(scale_cols):
    df[col] = cudf.Series(cp.asarray(scaled[:, i]), index=df.index)

print("Categorical ordinal columns:", len(ordinal_cols))
print("Yeo-Johnson candidates:", len(skewed_cols))
print("Signed-log fallbacks:", len(power_fallbacks))
print("Robust-scaled numerical features:", len(scale_cols))
print("Final GPU dataframe shape:", df.shape)


BracketError: The algorithm terminated without finding a valid bracket. Consider trying different initial points.

In [ ]:
if y is not None:
    df[TARGET] = y

# Export in row chunks to avoid materializing an extra full CPU copy.
CHUNK_SIZE = 25_000
if OUTPUT_PATH.exists(): OUTPUT_PATH.unlink()
total_rows = len(df)
for start in range(0, total_rows, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, total_rows)
    df.iloc[start:end].to_pandas().to_csv(
        OUTPUT_PATH, mode="w" if start == 0 else "a",
        header=(start == 0), index=False
    )
    print(f"Wrote rows {start:,}–{end:,} of {total_rows:,}")

ids.to_pandas().to_csv(IDS_OUTPUT_PATH, index=False)
print("Saved processed GPU-preprocessed data:", OUTPUT_PATH)
print("Saved TransactionID sidecar:", IDS_OUTPUT_PATH)


In [ ]:
# Optional diagnostics
print("Final shape:", df.shape)
print("Remaining missing cells:", int(df.isna().sum().sum()))
print("GPU memory in use:", round(cp.get_default_memory_pool().used_bytes() / 2**30, 3), "GB")
